In [ ]:
# cell 1
import pandas as pd
import numpy as np
from pathlib import Path

In [ ]:
# cell 2
POLICY_FILE = r"..\Data\raw data\OxCGRT_AUS_latest.csv"

OUTPUT_DIR = Path("../Data/processed_data")
OUTPUT_DIR.mkdir(exist_ok=True)

ROLLING_WINDOWS = 14

In [ ]:
# cell 3
# Load OxCGRT data

policy_raw = pd.read_csv(
    POLICY_FILE,
    low_memory=False
)

print("Raw policy shape:", policy_raw.shape)
print(policy_raw.columns.tolist())

Raw policy shape: (9864, 61)
['CountryName', 'CountryCode', 'RegionName', 'RegionCode', 'Jurisdiction', 'Date', 'C1M_School closing', 'C1M_Flag', 'C2M_Workplace closing', 'C2M_Flag', 'C3M_Cancel public events', 'C3M_Flag', 'C4M_Restrictions on gatherings', 'C4M_Flag', 'C5M_Close public transport', 'C5M_Flag', 'C6M_Stay at home requirements', 'C6M_Flag', 'C7M_Restrictions on internal movement', 'C7M_Flag', 'C8EV_International travel controls', 'E1_Income support', 'E1_Flag', 'E2_Debt/contract relief', 'E3_Fiscal measures', 'E4_International support', 'H1_Public information campaigns', 'H1_Flag', 'H2_Testing policy', 'H3_Contact tracing', 'H4_Emergency investment in healthcare', 'H5_Investment in vaccines', 'H6M_Facial Coverings', 'H6M_Flag', 'H7_Vaccination policy', 'H7_Flag', 'H8M_Protection of elderly people', 'H8M_Flag', 'M1_Wildcard', 'V1_Vaccine Prioritisation (summary)', 'V2A_Vaccine Availability (summary)', 'V2B_Vaccine age eligibility/availability age floor (general population s

In [ ]:
# cell 4
# Select policy and epidemic variables

selected_cols = [
    "RegionName",
    "RegionCode",
    "Date",
    "C1M_School closing",
    "C2M_Workplace closing",
    "H6M_Facial Coverings",
    "ConfirmedCases",
    "ConfirmedDeaths"
]

policy_df = policy_raw[selected_cols].copy()

print("Selected policy shape:", policy_df.shape)
print(policy_df.columns.tolist())

Selected policy shape: (9864, 8)
['RegionName', 'RegionCode', 'Date', 'C1M_School closing', 'C2M_Workplace closing', 'H6M_Facial Coverings', 'ConfirmedCases', 'ConfirmedDeaths']


In [ ]:
# cell 5
# Clean Date and basic rows

policy_df["Date"] = pd.to_datetime(
    policy_df["Date"],
    format="%Y%m%d",
    errors="coerce"
)

policy_df = policy_df.dropna(subset=["RegionName", "Date"]).copy()

policy_df = policy_df.sort_values(["RegionName", "Date"]).reset_index(drop=True)

print("Policy date range:", policy_df["Date"].min(), "to", policy_df["Date"].max())
print("Regions:")
print(policy_df["RegionName"].value_counts())

Policy date range: 2020-01-01 00:00:00 to 2022-12-31 00:00:00
Regions:
RegionName
Australian Capital Territory    1096
New South Wales                 1096
Northern Territory              1096
Queensland                      1096
South Australia                 1096
Tasmania                        1096
Victoria                        1096
Western Australia               1096
Name: count, dtype: int64


In [ ]:
# cell 5
# Check duplicates

duplicate_count = policy_df.duplicated(subset=["RegionName", "Date"]).sum()

print("Duplicate RegionName-Date rows:", duplicate_count)

if duplicate_count > 0:
    duplicates = policy_df[
        policy_df.duplicated(subset=["RegionName", "Date"], keep=False)
    ].sort_values(["RegionName", "Date"])
    display(duplicates.head(20))

Duplicate RegionName-Date rows: 0


In [ ]:
# cell 6
SURVEY_START_DATE = "2020-04-01"
SURVEY_END_DATE = "2022-03-28"

policy_df = policy_df[
    (policy_df["Date"] >= pd.to_datetime(SURVEY_START_DATE)) &
    (policy_df["Date"] <= pd.to_datetime(SURVEY_END_DATE))
].copy()

In [ ]:
# cell 7
# Missing value check for selected variables

missing_summary = pd.DataFrame({
    "variable": policy_df.columns,
    "missing_count": policy_df.isna().sum().values,
    "missing_rate": policy_df.isna().mean().values
}).sort_values("missing_rate", ascending=False)

missing_summary.to_csv(
    OUTPUT_DIR / "policy_missing_value_summary.csv",
    index=False
)

missing_summary

,variable,missing_count,missing_rate
0,RegionName,0,0.0
1,RegionCode,0,0.0
2,Date,0,0.0
3,C1M_School closing,0,0.0
4,C2M_Workplace closing,0,0.0
5,H6M_Facial Coverings,0,0.0
6,ConfirmedCases,0,0.0
7,ConfirmedDeaths,0,0.0


In [ ]:
# cell 8
# Check policy value distributions

policy_intensity_cols = [
    "C1M_School closing",
    "C2M_Workplace closing",
    "H6M_Facial Coverings"
]

for col in policy_intensity_cols:
    if col in policy_df.columns:
        print(f"\n{col}")
        print(policy_df[col].value_counts(dropna=False).sort_index())


C1M_School closing
C1M_School closing
0.0     419
1.0    4461
2.0     351
3.0     585
Name: count, dtype: int64

C2M_Workplace closing
C2M_Workplace closing
0.0     245
1.0    4269
2.0     572
3.0     730
Name: count, dtype: int64

H6M_Facial Coverings
H6M_Facial Coverings
0.0    1349
1.0     864
2.0    2543
3.0     558
4.0     502
Name: count, dtype: int64


In [ ]:
# cell 9
# Convert policy and epidemic variables to numeric

numeric_cols = [
    "C1M_School closing",
    "C2M_Workplace closing",
    "H6M_Facial Coverings",
    "ConfirmedCases",
    "ConfirmedDeaths",
    "C1M_Flag",
    "C2M_Flag",
    "H6M_Flag"
]

for col in numeric_cols:
    if col in policy_df.columns:
        policy_df[col] = pd.to_numeric(policy_df[col], errors="coerce")

In [ ]:
# cell 10
# Create daily cases and deaths by state

policy_df = policy_df.sort_values(["RegionName", "Date"]).copy()

policy_df["daily_cases"] = (
    policy_df
    .groupby("RegionName")["ConfirmedCases"]
    .diff()
)

policy_df["daily_deaths"] = (
    policy_df
    .groupby("RegionName")["ConfirmedDeaths"]
    .diff()
)

# First day in each region has no previous cumulative value
# Negative values may occur due to data corrections.
negative_cases = (policy_df["daily_cases"] < 0).sum()
negative_deaths = (policy_df["daily_deaths"] < 0).sum()

print("Negative daily_cases:", negative_cases)
print("Negative daily_deaths:", negative_deaths)

# Set negative daily differences to NaN and record them.
policy_df.loc[policy_df["daily_cases"] < 0, "daily_cases"] = np.nan
policy_df.loc[policy_df["daily_deaths"] < 0, "daily_deaths"] = np.nan

Negative daily_cases: 92
Negative daily_deaths: 14


In [ ]:
# cell 11
# Create rolling epidemic severity indicators

policy_df["cases_14d_avg"] = (
    policy_df
    .groupby("RegionName")["daily_cases"]
    .transform(lambda x: x.rolling(window=ROLLING_WINDOWS, min_periods=1).mean())
)

policy_df["deaths_14d_avg"] = (
    policy_df
    .groupby("RegionName")["daily_deaths"]
    .transform(lambda x: x.rolling(window=ROLLING_WINDOWS, min_periods=1).mean())
)

In [ ]:
# cell 12
core_policy_cols = [
    "RegionName",
    "RegionCode",
    "Date",
    "C1M_School closing",
    "C2M_Workplace closing",
    "H6M_Facial Coverings",
    "cases_14d_avg",
    "deaths_14d_avg"
]


policy_df = policy_df.dropna(subset=core_policy_cols).copy()


In [ ]:
# cell 13
# Final check and save

final_missing_summary = pd.DataFrame({
    "variable": policy_df.columns,
    "missing_count": policy_df.isna().sum().values,
    "missing_rate": policy_df.isna().mean().values
}).sort_values("missing_rate", ascending=False)

final_missing_summary.to_csv(
    OUTPUT_DIR / "policy_missing_value_summary_after_cleaning.csv",
    index=False
)

print("Final policy shape:", policy_df.shape)
print("Final date range:", policy_df["Date"].min(), "to", policy_df["Date"].max())

policy_df.to_csv(
    OUTPUT_DIR / "cleaned_policy_epidemic.csv",
    index=False
)

final_missing_summary.head(20)

Final policy shape: (5808, 12)
Final date range: 2020-04-02 00:00:00 to 2022-03-28 00:00:00


,variable,missing_count,missing_rate
8,daily_cases,92,0.01584
9,daily_deaths,14,0.00241
0,RegionName,0,0.00000
1,RegionCode,0,0.00000
2,Date,0,0.00000
3,C1M_School closing,0,0.00000
4,C2M_Workplace closing,0,0.00000
5,H6M_Facial Coverings,0,0.00000
6,ConfirmedCases,0,0.00000
7,ConfirmedDeaths,0,0.00000
